# Area Per Lipid Analysis

This notebook calculates area per lipid (APL) from
all-atom molecular dynamics simulations of amyloid-β
peptides interacting with DMPC and cholesterol lipids.

The analysis includes:

- Identification of lipid leaflets using the nematic director
- Projection of lipid headgroup coordinates onto the membrane plane
- Calculation of projected leaflet areas using convex hulls
- Calculation of mean APL and SEM across independent trajectories

The analysis considers Aβ40 and Aβ42 hexamers with free lipids and 
lipid-only controls.

In [1]:

import mdtraj as md
import numpy as np
from mdtraj.geometry.order import compute_directors


def n_director(traj, indices="chains"):
    """
    Calculate the principal eigenvectors of the nematic ordering
    tensor (Q) for each trajectory frame.

    Returns
    -------
    largest_eigenvectors : ndarray, shape (n_frames, 3)
        Eigenvectors corresponding to the largest eigenvalues.
    middle_eigenvectors : ndarray, shape (n_frames, 1, 3)
        Eigenvectors corresponding to the middle eigenvalues.
    smallest_eigenvectors : ndarray, shape (n_frames, 3)
        Eigenvectors corresponding to the smallest eigenvalues.
    """

    # Calculate molecular directors and the nematic ordering tensor.
    all_directors = compute_directors(traj, indices)
    Q_ab = md.geometry.order._compute_Q_tensor(all_directors)

    largest_eigenvectors = []
    middle_eigenvectors = []
    smallest_eigenvectors = []

    # Diagonalize the Q-tensor for each frame.
    for i in range(Q_ab.shape[0]):

        eigenvalues, eigenvectors = np.linalg.eigh(Q_ab[i])

        sorted_indices = np.argsort(eigenvalues)
        sorted_eigenvectors = eigenvectors[:, sorted_indices]

        largest_eigenvectors.append(sorted_eigenvectors[:, -1])

        if sorted_eigenvectors.shape[1] > 2:
            middle_eigenvectors.append(sorted_eigenvectors[:, 1:-1].T)

        smallest_eigenvectors.append(sorted_eigenvectors[:, 0])

    largest_eigenvectors = np.array(largest_eigenvectors)
    middle_eigenvectors = np.array(middle_eigenvectors)
    smallest_eigenvectors = np.array(smallest_eigenvectors)

    return largest_eigenvectors, middle_eigenvectors, smallest_eigenvectors


## Separate leaflets

In [2]:

def split_leaflets(xtc_file, gro_file, start_ns=None, end_ns=None):
    """
    Separate lipid molecules into upper and lower membrane leaflets
    based on their positions relative to the collective nematic director.

    DMPC nitrogen (N) and cholesterol oxygen (O3) atoms are used
    to represent the positions of the lipid molecules.

    Parameters
    ----------
    xtc_file : str
        Path to the trajectory file (.xtc).
    gro_file : str
        Path to the topology file (.gro).
    start_ns, end_ns : float, optional
        Analysis time interval in nanoseconds.

    Returns
    -------
    dict
        Coordinates and lipid types for both leaflets in each frame,
        along with the collective nematic directors.
    """

    traj = md.load(xtc_file, top=gro_file)

    # Select the analysis time interval.
    if start_ns is not None or end_ns is not None:

        time_ns = traj.time / 1000.0
        mask = np.ones(traj.n_frames, dtype=bool)

        if start_ns is not None:
            mask = mask & (time_ns >= start_ns)

        if end_ns is not None:
            mask = mask & (time_ns <= end_ns)

        if np.sum(mask) == 0:
            raise ValueError(
                f"No frames found between {start_ns} and {end_ns} ns. "
                f"Trajectory time range is {time_ns.min()} to {time_ns.max()} ns."
            )

        traj = traj[mask]

    top = traj.topology

    # Select representative headgroup atoms for DMPC and CHOL.
    dmpc_idx = top.select("resname DMPC and name N")
    chol_idx = top.select("resname CHL1 and name O3")

    lipid_idx_list = []
    lipid_type_list = []

    if len(dmpc_idx) > 0:
        lipid_idx_list.append(dmpc_idx)
        lipid_type_list += ["DMPC"] * len(dmpc_idx)

    if len(chol_idx) > 0:
        lipid_idx_list.append(chol_idx)
        lipid_type_list += ["CHL1"] * len(chol_idx)

    if len(lipid_idx_list) == 0:
        raise ValueError("No DMPC N or CHL1 O3 lipid head atoms found.")

    lipid_idx = np.concatenate(lipid_idx_list).astype(int)
    lipid_types = np.array(lipid_type_list)

    lipid_xyz = traj.xyz[:, lipid_idx, :]

    # Calculate the collective nematic director for each frame.
    directors, _, _ = n_director(traj)

    top_xyz = []
    bottom_xyz = []
    top_types = []
    bottom_types = []

    # Classify lipids based on their positions along the director.
    for t in range(traj.n_frames):

        n = directors[t]
        pos = lipid_xyz[t]

        center = pos.mean(axis=0)
        rel = pos - center

        z_proj = rel @ n

        mask_top = z_proj > 0
        mask_bottom = z_proj < 0

        top_xyz.append(pos[mask_top])
        bottom_xyz.append(pos[mask_bottom])

        top_types.append(lipid_types[mask_top])
        bottom_types.append(lipid_types[mask_bottom])

    return {
        "top_xyz": top_xyz,
        "bottom_xyz": bottom_xyz,
        "top_types": top_types,
        "bottom_types": bottom_types,
        "directors": directors
    }


## Projecting coordinates into bilayer midplane (normal to director)

In [3]:

def project_to_membrane_plane(leaflet_xyz, directors):
    """
    Project lipid headgroup coordinates onto a 2D plane perpendicular
    to the collective nematic director for each trajectory frame.

    Parameters
    ----------
    leaflet_xyz : list of ndarray
        3D lipid headgroup coordinates for each frame.
    directors : ndarray
        Collective nematic director for each frame.

    Returns
    -------
    projected_xy : list of ndarray
        2D projected coordinates for each frame.
    """

    projected_xy = []

    for t in range(len(leaflet_xyz)):

        pos = leaflet_xyz[t]

        if len(pos) < 3:
            projected_xy.append(np.empty((0, 2)))
            continue

        n = directors[t] / np.linalg.norm(directors[t])

        # Construct two orthogonal basis vectors in the membrane plane.
        ref = np.array([1.0, 0.0, 0.0])
        if abs(np.dot(ref, n)) > 0.9:
            ref = np.array([0.0, 1.0, 0.0])

        e1 = np.cross(n, ref)
        e1 /= np.linalg.norm(e1)

        e2 = np.cross(n, e1)
        e2 /= np.linalg.norm(e2)

        # Center coordinates and project onto the 2D basis.
        center = pos.mean(axis=0)
        rel = pos - center

        u = rel @ e1
        v = rel @ e2

        xy = np.column_stack((u, v))
        projected_xy.append(xy)

    return projected_xy


In [4]:

from scipy.spatial import ConvexHull


def leaflet_area(xy):
    """
    Calculate the area enclosed by the 2D convex hull of
    projected lipid headgroup coordinates.

    Parameters
    ----------
    xy : ndarray, shape (n_lipids, 2)
        Projected lipid headgroup coordinates in nm.

    Returns
    -------
    area : float
        Convex hull area in nm². Returns NaN if fewer than
        three points are available.
    """

    if xy is None or len(xy) < 3:
        return np.nan

    hull = ConvexHull(xy)

    # For a 2D convex hull, volume represents the enclosed area.
    area = hull.volume

    return area


In [5]:

import pandas as pd


def calculate_apl_one_trajectory(xtc_file, gro_file, start_ns=None, end_ns=None):
    """
    Calculate the mean area per lipid (APL) for a single MD trajectory.

    The convex hull area of each leaflet is divided by its lipid count.
    The two leaflet APL values are averaged for each frame, followed
    by averaging over all selected frames.

    Parameters
    ----------
    xtc_file, gro_file : str
        Paths to trajectory (.xtc) and topology (.gro) files.
    start_ns, end_ns : float, optional
        Analysis time interval in nanoseconds.

    Returns
    -------
    traj_mean_apl : float
        Mean area per lipid across selected frames (nm²/lipid).
    df : pandas.DataFrame
        Frame-by-frame leaflet areas, lipid counts, and APL values.
    """

    leaflets = split_leaflets(xtc_file, gro_file, start_ns=start_ns, end_ns=end_ns)

    top_xy = project_to_membrane_plane(leaflets["top_xyz"], leaflets["directors"])
    bottom_xy = project_to_membrane_plane(leaflets["bottom_xyz"], leaflets["directors"])

    frame_results = []

    for t in range(len(top_xy)):

        top_area = leaflet_area(top_xy[t])
        bottom_area = leaflet_area(bottom_xy[t])

        n_top = len(top_xy[t])
        n_bottom = len(bottom_xy[t])

        top_apl = top_area / n_top if n_top > 0 else np.nan
        bottom_apl = bottom_area / n_bottom if n_bottom > 0 else np.nan

        mean_apl = np.nanmean([top_apl, bottom_apl])

        frame_results.append({
            "frame": t,
            "top_area": top_area,
            "bottom_area": bottom_area,
            "n_top": n_top,
            "n_bottom": n_bottom,
            "top_apl": top_apl,
            "bottom_apl": bottom_apl,
            "mean_apl": mean_apl
        })

    df = pd.DataFrame(frame_results)

    traj_mean_apl = df["mean_apl"].mean()

    return traj_mean_apl, df


In [6]:

def calculate_apl_multiple_trajectories(traj_list, top_list, start_ns=None, end_ns=None):
    """
    Calculate the mean APL and SEM across independent MD trajectories.

    Each trajectory contributes one time-averaged APL value.
    The final mean and SEM are calculated from these trajectory means.

    Parameters
    ----------
    traj_list, top_list : list of str
        Paths to trajectory (.xtc) and topology (.gro) files.
    start_ns, end_ns : float, optional
        Analysis time interval in nanoseconds.

    Returns
    -------
    final_mean : float
        Mean APL across trajectories (nm²/lipid).
    final_sem : float
        Standard error of the mean (nm²/lipid).
    summary_df : pandas.DataFrame
        Mean APL for each independent trajectory.
    all_frames_df : pandas.DataFrame
        Frame-by-frame APL results from all trajectories.
    """

    traj_values = []
    all_frame_dfs = []

    for i, (xtc_file, gro_file) in enumerate(zip(traj_list, top_list), start=1):

        print(f"Processing trajectory {i}")

        traj_mean_apl, frame_df = calculate_apl_one_trajectory(
            xtc_file, gro_file, start_ns=start_ns, end_ns=end_ns
        )

        traj_values.append(traj_mean_apl)

        frame_df["trajectory"] = i
        all_frame_dfs.append(frame_df)

    traj_values = np.array(traj_values)

    # Calculate mean APL and SEM across independent trajectories.
    final_mean = np.nanmean(traj_values)

    if len(traj_values) > 1:
        final_sem = np.nanstd(traj_values, ddof=1) / np.sqrt(len(traj_values))
    else:
        final_sem = np.nan

    summary_df = pd.DataFrame({
        "trajectory": np.arange(1, len(traj_values) + 1),
        "mean_apl": traj_values
    })

    all_frames_df = pd.concat(all_frame_dfs, ignore_index=True)

    return final_mean, final_sem, summary_df, all_frames_df


In [7]:

from pathlib import Path

# Base directory containing simulation trajectories
BASE_DIR = Path("/data/B/rpandey/ANALYSIS/AB_lipids_1_5_ratio/xtc_files_for_movies")

def get_files(folder, n):
    path = BASE_DIR / folder

    traj = [str(path / f"movie_smooth_{i:02d}.xtc") for i in range(1, n + 1)]
    top = [str(path / f"movie_smooth_{i:02d}.gro") for i in range(1, n + 1)]

    return traj, top


# 30 DMPC systems
dmpc30_traj, dmpc30_top = get_files("30DMPC_Control", 3)
ab40_30dmpc_traj, ab40_30dmpc_top = get_files("AB40_six_monomers_30DMPC", 5)
ab42_30dmpc_traj, ab42_30dmpc_top = get_files("AB42_six_monomers_30DMPC", 5)

# 21 DMPC + 9 CHOL systems
dmpc21_9chl1_traj, dmpc21_9chl1_top = get_files("21DMPC_9CHL1_Control", 3)
ab40_21dmpc_9chl1_traj, ab40_21dmpc_9chl1_top = get_files("AB40_six_monomers_21DMPC_9CHL1", 5)
ab42_21dmpc_9chl1_traj, ab42_21dmpc_9chl1_top = get_files("AB42_six_monomers_21DMPC_9CHL1", 5)


In [8]:

# ============================================================
# 30 DMPC
# ============================================================

# Lipid-only control
final_mean_dmpc30, final_sem_dmpc30, summary_dmpc30, frames_dmpc30 = calculate_apl_multiple_trajectories(
    dmpc30_traj, dmpc30_top, start_ns=900, end_ns=1000
)

# Aβ40 + 30 DMPC
final_mean_ab40_30dmpc, final_sem_ab40_30dmpc, summary_ab40_30dmpc, frames_ab40_30dmpc = calculate_apl_multiple_trajectories(
    ab40_30dmpc_traj, ab40_30dmpc_top, start_ns=900, end_ns=1000
)

# Aβ42 + 30 DMPC
final_mean_ab42_30dmpc, final_sem_ab42_30dmpc, summary_ab42_30dmpc, frames_ab42_30dmpc = calculate_apl_multiple_trajectories(
    ab42_30dmpc_traj, ab42_30dmpc_top, start_ns=900, end_ns=1000
)


# ============================================================
# 21 DMPC + 9 CHOL
# ============================================================

# Lipid-only control
final_mean_dmpc21_9chl1, final_sem_dmpc21_9chl1, summary_dmpc21_9chl1, frames_dmpc21_9chl1 = calculate_apl_multiple_trajectories(
    dmpc21_9chl1_traj, dmpc21_9chl1_top, start_ns=900, end_ns=1000
)

# Aβ40 + 21 DMPC + 9 CHOL
final_mean_ab40_21dmpc_9chl1, final_sem_ab40_21dmpc_9chl1, summary_ab40_21dmpc_9chl1, frames_ab40_21dmpc_9chl1 = calculate_apl_multiple_trajectories(
    ab40_21dmpc_9chl1_traj, ab40_21dmpc_9chl1_top, start_ns=900, end_ns=1000
)

# Aβ42 + 21 DMPC + 9 CHOL
final_mean_ab42_21dmpc_9chl1, final_sem_ab42_21dmpc_9chl1, summary_ab42_21dmpc_9chl1, frames_ab42_21dmpc_9chl1 = calculate_apl_multiple_trajectories(
    ab42_21dmpc_9chl1_traj, ab42_21dmpc_9chl1_top, start_ns=900, end_ns=1000
)


Processing trajectory 1
Processing trajectory 2
Processing trajectory 3
Processing trajectory 1
Processing trajectory 2
Processing trajectory 3
Processing trajectory 4
Processing trajectory 5
Processing trajectory 1
Processing trajectory 2
Processing trajectory 3
Processing trajectory 4
Processing trajectory 5
Processing trajectory 1
Processing trajectory 2
Processing trajectory 3
Processing trajectory 1
Processing trajectory 2
Processing trajectory 3
Processing trajectory 4
Processing trajectory 5
Processing trajectory 1
Processing trajectory 2
Processing trajectory 3
Processing trajectory 4
Processing trajectory 5


In [9]:

# Mean APL, SEM, and individual trajectory results
results = [
    ("30 DMPC Control", final_mean_dmpc30, final_sem_dmpc30, summary_dmpc30),
    ("Aβ40 + 30 DMPC", final_mean_ab40_30dmpc, final_sem_ab40_30dmpc, summary_ab40_30dmpc),
    ("Aβ42 + 30 DMPC", final_mean_ab42_30dmpc, final_sem_ab42_30dmpc, summary_ab42_30dmpc),
    ("21 DMPC + 9 CHOL Control", final_mean_dmpc21_9chl1, final_sem_dmpc21_9chl1, summary_dmpc21_9chl1),
    ("Aβ40 + 21 DMPC + 9 CHOL", final_mean_ab40_21dmpc_9chl1, final_sem_ab40_21dmpc_9chl1, summary_ab40_21dmpc_9chl1),
    ("Aβ42 + 21 DMPC + 9 CHOL", final_mean_ab42_21dmpc_9chl1, final_sem_ab42_21dmpc_9chl1, summary_ab42_21dmpc_9chl1)
]

for label, mean, sem, summary in results:
    print(f"\n{label}")
    print(f"Mean APL: {mean} nm²/lipid")
    print(f"SEM: {sem} nm²/lipid")

    print("APL per trajectory:")
    for _, row in summary.iterrows():
        print(f"  Trajectory {int(row['trajectory'])}: {row['mean_apl']} nm²/lipid")

    print("-" * 50)



30 DMPC Control
Mean APL: 0.8076310621373182 nm²/lipid
SEM: 0.001841646589627162 nm²/lipid
APL per trajectory:
  Trajectory 1: 0.8040024810761262 nm²/lipid
  Trajectory 2: 0.8088975971613015 nm²/lipid
  Trajectory 3: 0.8099931081745271 nm²/lipid
--------------------------------------------------

Aβ40 + 30 DMPC
Mean APL: 0.9317186196419455 nm²/lipid
SEM: 0.03657279581279862 nm²/lipid
APL per trajectory:
  Trajectory 1: 0.9945835221559907 nm²/lipid
  Trajectory 2: 0.9848787612771855 nm²/lipid
  Trajectory 3: 0.7922506901434373 nm²/lipid
  Trajectory 4: 0.9545599420183513 nm²/lipid
  Trajectory 5: 0.9323201826147623 nm²/lipid
--------------------------------------------------

Aβ42 + 30 DMPC
Mean APL: 0.8966752900417714 nm²/lipid
SEM: 0.043909122154547425 nm²/lipid
APL per trajectory:
  Trajectory 1: 0.7813122499073956 nm²/lipid
  Trajectory 2: 0.9196216977245811 nm²/lipid
  Trajectory 3: 1.0154512873494599 nm²/lipid
  Trajectory 4: 0.8118370215307329 nm²/lipid
  Trajectory 5: 0.9551541